# Лабораторная работа №1
## Методы агрегации и группировки данных

**Выполнил:** Бушуев Никита  
**Вариант 1.** Набор данных: drivers.csv

---

**Цель работы:** овладение навыками агрегации и группировки табличных данных для преобразования детальных записей в структурированные аналитические отчёты

## Загрузка данных

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('drivers.csv', sep=';')
print("Размер датасета:", df.shape)
print("Столбцы:", df.columns.tolist())
df.info()
df.head(20)

Размер датасета: (161, 7)
Столбцы: ['START_DATE', 'END_DATE', 'CATEGORY*', 'START', 'STOP', 'MILES', 'PURPOSEroute']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   START_DATE    161 non-null    object 
 1   END_DATE      161 non-null    object 
 2   CATEGORY*     161 non-null    object 
 3   START         161 non-null    object 
 4   STOP          161 non-null    object 
 5   MILES         161 non-null    float64
 6   PURPOSEroute  84 non-null     object 
dtypes: float64(1), object(6)
memory usage: 8.9+ KB


,START_DATE,END_DATE,CATEGORY*,START,STOP,MILES,PURPOSEroute
0,01.10.2016 19:12,01.10.2016 19:32,Business,Midtown,East Harlem,44963.0,MEETING
1,01.11.2016 13:32,01.11.2016 13:46,Business,Midtown,Midtown East,45108.0,Meal/Entertain
2,01.12.2016 12:33,01.12.2016 12:49,Business,Midtown,Hudson Square,45170.0,Meal/Entertain
3,1.13.2016 15:00,1.13.2016 15:28,Business,Gulfton,Downtown,45149.0,Meeting
4,1.29.2016 21:21,1.29.2016 21:40,Business,Apex,Cary,45051.0,Meal/Entertain
5,1.30.2016 18:09,1.30.2016 18:24,Business,Apex,Cary,45112.0,Customer Visit
6,02.01.2016 12:10,02.01.2016 12:43,Business,Chapel Hill,Cary,45008.0,Customer Visit
7,02.04.2016 9:37,02.04.2016 10:09,Business,Morrisville,Cary,45116.0,Meal/Entertain
8,02.07.2016 18:03,02.07.2016 18:17,Business,Apex,Cary,45112.0,Customer Visit
9,02.07.2016 20:22,02.07.2016 20:40,Business,Morrisville,Cary,44932.0,Meeting


## Описание предметной области и столбцов

Датасет `drivers.csv` - журнал поездок на такси. Каждая строка - одна поездка. 7 столбцов:

| Столбец | Описание |
|--------|---------|
| `START_DATE` | Дата и время начала поездки |
| `END_DATE` | Дата и время окончания поездки |
| `CATEGORY*` | Категория: Business (деловая) или Personal (личная) |
| `START` | Место отправления |
| `STOP` | Место назначения |
| `MILES` | Показания одометра (мили) |
| `PURPOSEroute` | Цель поездки (Meeting, Meal/Entertain, Customer Visit и др.) |

Необходимо выполнить:
- **Переименование столбцов**: убираем спецсимвол `*` из `CATEGORY*` и заменяем `route` в `PURPOSEroute`
- **Преобразование дат** в тип `datetime`
- **Расчёт столбца `DURATION`** - время поездки в минутах
- **Исправление неявных дубликатов** в `CATEGORY` и `PURPOSE_ROUTE`: `BUSINESS` - `Business`, `MEETING` - `Meeting`
- **Заполнение пропусков** в `PURPOSE_ROUTE`: 77 значений `NaN` - `'не определена'`

In [3]:
df.rename(columns={'CATEGORY*': 'CATEGORY', 'PURPOSEroute': 'PURPOSE_ROUTE'}, inplace=True)

df['START_DATE'] = pd.to_datetime(df['START_DATE'], format='mixed', dayfirst=False)
df['END_DATE'] = pd.to_datetime(df['END_DATE'], format='mixed', dayfirst=False)

df['DURATION'] = (df['END_DATE'] - df['START_DATE']).dt.total_seconds() / 60

df['CATEGORY'] = df['CATEGORY'].str.strip().str.title()
df['PURPOSE_ROUTE'] = df['PURPOSE_ROUTE'].str.strip().str.title()

df['PURPOSE_ROUTE'] = df['PURPOSE_ROUTE'].fillna('не определена')

print("Уникальные значения CATEGORY:", df['CATEGORY'].unique())
print("\nPURPOSE_ROUTE после очистки:")
print(df['PURPOSE_ROUTE'].value_counts())
print("\nПропуски:", df.isnull().sum().sum())

Уникальные значения CATEGORY: ['Business' 'Personal']

PURPOSE_ROUTE после очистки:
PURPOSE_ROUTE
не определена     77
Meal/Entertain    36
Customer Visit    30
Meeting           13
Temporary Site     4
Moving             1
Name: count, dtype: int64

Пропуски: 0


## Задание 1: Группировка по CATEGORY и PURPOSE_ROUTE

Группировка по двум столбцам: `CATEGORY` (тип поездки) и `PURPOSE_ROUTE` (цель маршрута). Результат - количество поездок каждого типа по каждой цели.

In [4]:
result1 = df.groupby(['CATEGORY', 'PURPOSE_ROUTE'])['START_DATE'].count()
print("Задание 1 - количество поездок по категории и цели маршрута:")
print(result1)

Задание 1 - количество поездок по категории и цели маршрута:
CATEGORY  PURPOSE_ROUTE 
Business  Customer Visit    30
          Meal/Entertain    36
          Meeting           13
          Temporary Site     4
          не определена     67
Personal  Moving             1
          не определена     10
Name: START_DATE, dtype: int64


**Вывод:**
Большинство поездок (150 из 161) деловые (Business), личных поездок всего 11.Среди деловых поездок наиболее частые цели Meal/Entertain (36) и Customer Visit (30). У 67 деловых поездок цель маршрута не указана (не определена). Среди личных поездок единственная указанная цель - Moving (1 поездка), у 10 поездок цель не определена

## Задание 2: Группировка по CATEGORY и точке старта START

Группировка по `CATEGORY` и `START` с подсчётом количества поездок. Создаём DataFrame, переименовываем столбец в `count`, сортируем по **возрастанию** `count`.

In [5]:
result2 = df.groupby(['CATEGORY', 'START'])['END_DATE'].count().reset_index()
result2.columns = ['CATEGORY', 'START', 'count']
result2 = result2.sort_values('count', ascending=True)

print("Задание 2 - количество поездок по категории и точке старта:")
print(result2.to_string(index=False))

Задание 2 - количество поездок по категории и точке старта:
CATEGORY                 START  count
Business                Almond      1
Business                 Arabi      1
Business          Briar Meadow      1
Business             Arlington      1
Business        Georgian Acres      1
Business               Gulfton      1
Business      Columbia Heights      1
Business        College Avenue      1
Business            Hayesville      1
Business           Santa Clara      1
Personal           Chessington      1
Personal                 Boone      1
Personal               Midtown      1
Business        South Berkeley      1
Business               Marigny      1
Business Lower Garden District      1
Personal              Sky Lake      1
Personal     Sand Lake Commons      1
Business              San Jose      2
Business          Savon Height      2
Business           Capitol One      2
Business             Galveston      2
Business                 South      2
Business              SOMISS

**Вывод:** Самая популярная точка отправления для Business поездок - Morrisville (70 поездок). Для Personal поездок - Morrisville (6 поездок). Большинство точек - единичные поездки.

## Задание 3: Сводная таблица - среднее MILES по PURPOSE_ROUTE

`pivot_table` - среднее количество пройденных миль по каждой цели поездки (`PURPOSE_ROUTE`).  
Сортировка по убыванию `MILES`, округление до 2 знаков.

In [6]:
result3 = df.pivot_table(
    index='PURPOSE_ROUTE',
    values='MILES',
    aggfunc='mean'
).round(2)

result3 = result3.sort_values('MILES', ascending=False)
print("Задание 3 — среднее MILES по цели поездки:")
print(result3)

Задание 3 — среднее MILES по цели поездки:
                   MILES
PURPOSE_ROUTE           
Temporary Site  45063.50
Moving          44932.00
Meal/Entertain  41276.67
не определена   36859.20
Customer Visit  36023.20
Meeting         34646.85


**Вывод:** Наибольшее среднее значение у поездок с целью Temporary Site (45 063). Это объясняется тем, что данные в столбце `MILES` - накопленные показания, и для поздних поездок (в конце года) они выше.

## Задание 4: Сводная таблица — среднее MILES по CATEGORY и START

`pivot_table` - среднее количество пройденных миль, где:
- **столбцы** = `CATEGORY`  
- **строки** = `START` (точка старта)  

Сортировка по убыванию `Business`, округление через `.round()`.

In [7]:
result4 = df.pivot_table(
    index='START',
    columns='CATEGORY',
    values='MILES',
    aggfunc='mean'
).round(2)

result4 = result4.sort_index(ascending=False)
print("Задание 4 - среднее MILES по точке старта и категории:")
print(result4.to_string())

Задание 4 - среднее MILES по точке старта и категории:
CATEGORY               Business  Personal
START                                    
South Berkeley             0.90       NaN
South                  22511.40       NaN
Sky Lake                    NaN       6.0
Savon Height           45065.50       NaN
Santa Clara               43.90       NaN
Sand Lake Commons           NaN   44963.0
San Jose               22592.80       NaN
SOMISSPO               45017.50       NaN
Morrisville            39882.89   45001.5
Midtown                38117.08       1.0
Metairie               45061.00       NaN
Marigny                45047.00       NaN
Mandeville             22585.00       NaN
Lower Garden District  45022.00       NaN
Hayesville                75.70       NaN
Gulfton                45149.00       NaN
Georgian Acres         44963.00       NaN
Galveston              22493.00       NaN
Columbia Heights       45047.00       NaN
Colombo                39431.75       NaN
College Avenue       

**Вывод:** Точки старта с наибольшим средним значением одометра (Arlington, Gulfton, Briar Meadow ~45 000–45 173)  
— это поздние поездки года. Реальные малые расстояния (South Berkeley = 0.9) отражают короткие поездки с малым пробегом.

## Задание 5: Категория длительности поездки и сводная таблица

**Аргументация выбора границ категорий:**  
На основе `describe()`: 25-й перцентиль = 9 мин, медиана = 15 мин, 75-й перцентиль = 22 мин.  
- **Короткая**: до 9 минут - поездки короче 1-го квартиля (25% всех поездок)  
- **Средняя**: 9–22 минуты - поездки в межквартильном диапазоне (50% всех поездок)  
- **Длинная**: более 22 минут - поездки выше 3-го квартиля (25% всех поездок)

Создаём столбец `DURATION_CATEGORY` с помощью `pd.cut()` (3 категории).
Сводная таблица: среднее `MILES` по `DURATION_CATEGORY` и `PURPOSE_ROUTE`.

In [8]:
print("Статистика DURATION для выбора границ категорий:")
print(df['DURATION'].describe().round(2))

bins_dur = [0, 9, 22, float('inf')]
labels_dur = ['Короткая', 'Средняя', 'Длинная']
df['DURATION_CATEGORY'] = pd.cut(df['DURATION'], bins=bins_dur, labels=labels_dur)

print("\nРаспределение по DURATION_CATEGORY:")
print(df['DURATION_CATEGORY'].value_counts().sort_index())

result5 = df.pivot_table(
    index='DURATION_CATEGORY',
    columns='PURPOSE_ROUTE',
    values='MILES',
    aggfunc='mean',
    observed=True
).round(2)

print("\nЗадание 5 - среднее MILES по DURATION_CATEGORY и PURPOSE_ROUTE:")
print(result5.to_string())

Статистика DURATION для выбора границ категорий:
count    161.00
mean      20.04
std       21.72
min        1.00
25%        9.00
50%       15.00
75%       22.00
max      206.00
Name: DURATION, dtype: float64

Распределение по DURATION_CATEGORY:
DURATION_CATEGORY
Короткая    42
Средняя     79
Длинная     40
Name: count, dtype: int64

Задание 5 - среднее MILES по DURATION_CATEGORY и PURPOSE_ROUTE:
PURPOSE_ROUTE      Customer Visit  Meal/Entertain   Meeting   Moving  Temporary Site  не определена
DURATION_CATEGORY                                                                                  
Короткая                 22512.50        42380.94       NaN      NaN             NaN       36470.56
Средняя                  40278.21        45026.00  30015.89  44932.0         45035.5       39886.17
Длинная                  32194.27        30024.33  45066.50      NaN         45091.5       32202.90


**Вывод:** Наибольшая группа - поездки средней длительности (9–22 мин, 79 записей),
они охватывают все цели маршрута. Длинные поездки (более 22 мин) на Meeting (45 067) и Temporary Site (45 092) имеют наибольшие средние значения одометра - это поздние поездки года с накопленным пробегом. Короткие поездки (до 9 мин) на Customer Visit показывают значительно меньшее среднее MILES (22 513) - это ранние поездки начала года. Для Meal/Entertain наблюдается обратная зависимость: у коротких поездок одометр выше (42 381), чем у длинных (30 024), что указывает на разный период совершения поездок.

## Задание 6: Категория пройденных миль и группировка

Создаём столбец `MILES_CATEGORY` с помощью `pd.cut()` (3 категории):

**Аргументация выбора границ:**  
На основе `describe()`: столбец `MILES` содержит накопленные показания одометра.  
Большинство значений сосредоточено в диапазоне 44 931–45 177 (25–75 перцентили),  
поэтому равномерные интервалы не подходят - почти все записи попали бы в одну категорию.  
Разбиваем по 33-му (44 959) и 66-му (45 050) перцентилям - это даёт три примерно равные группы:
- **Короткая** - одометр <= 44 959 (56 поездок - ранние поездки года, меньший накопленный пробег)
- **Средняя** - одометр 44 959–45 050 (51 поездка - середина года)
- **Длинная** - одометр > 45 050 (54 поездки - поздние поездки года, наибольший накопленный пробег)

Группировка: минимальная и максимальная длительность поездки по `MILES_CATEGORY`

In [9]:
print("Статистика MILES для выбора границ категорий:")
print(df['MILES'].describe().round(2))

q33 = df['MILES'].quantile(0.33)
q66 = df['MILES'].quantile(0.66)
print(f"\n33-й перцентиль: {q33:.0f}, 66-й перцентиль: {q66:.0f}")

bins_miles  = [0, q33, q66, float('inf')]
labels_miles = ['Короткая', 'Средняя', 'Длинная']
df['MILES_CATEGORY'] = pd.cut(df['MILES'], bins=bins_miles, labels=labels_miles)

print("\nРаспределение по MILES_CATEGORY:")
print(df['MILES_CATEGORY'].value_counts().sort_index())

result6 = df.groupby('MILES_CATEGORY', observed=True)['DURATION'].agg(['min', 'max']).round(2)
result6.columns = ['min_DURATION', 'max_DURATION']
print("\nЗадание 6 — мин и макс DURATION по MILES_CATEGORY:")
print(result6)

Статистика MILES для выбора границ категорий:
count      161.00
mean     37766.52
std      16614.93
min          0.80
25%      44931.00
50%      45008.00
75%      45081.00
max      45177.00
Name: MILES, dtype: float64

33-й перцентиль: 44959, 66-й перцентиль: 45050

Распределение по MILES_CATEGORY:
MILES_CATEGORY
Короткая    56
Средняя     51
Длинная     54
Name: count, dtype: int64

Задание 6 — мин и макс DURATION по MILES_CATEGORY:
                min_DURATION  max_DURATION
MILES_CATEGORY                            
Короткая                 1.0         206.0
Средняя                  5.0          62.0
Длинная                  2.0          88.0


**Вывод:** Поездки категории «Короткая» (ранние поездки года) имеют наибольший разброс длительности - от 1 до 206 минут: среди них есть как мгновенные поездки, так и самая длинная в датасете. Это говорит о том, что показания одометра не определяют длительность поездки - маршруты с малым пробегом могут занимать разное время. «Средняя» категория наиболее однородна по длительности: максимум составляет 62 минуты против 88 и 206 у остальных групп

## Итоговый датасет

In [10]:
print("Итоговый датасет — первые 20 строк:")
df.head(20)

Итоговый датасет — первые 20 строк:


,START_DATE,END_DATE,CATEGORY,START,STOP,MILES,PURPOSE_ROUTE,DURATION,DURATION_CATEGORY,MILES_CATEGORY
0,2016-01-10 19:12:00,2016-01-10 19:32:00,Business,Midtown,East Harlem,44963.0,Meeting,20.0,Средняя,Средняя
1,2016-01-11 13:32:00,2016-01-11 13:46:00,Business,Midtown,Midtown East,45108.0,Meal/Entertain,14.0,Средняя,Длинная
2,2016-01-12 12:33:00,2016-01-12 12:49:00,Business,Midtown,Hudson Square,45170.0,Meal/Entertain,16.0,Средняя,Длинная
3,2016-01-13 15:00:00,2016-01-13 15:28:00,Business,Gulfton,Downtown,45149.0,Meeting,28.0,Длинная,Длинная
4,2016-01-29 21:21:00,2016-01-29 21:40:00,Business,Apex,Cary,45051.0,Meal/Entertain,19.0,Средняя,Длинная
5,2016-01-30 18:09:00,2016-01-30 18:24:00,Business,Apex,Cary,45112.0,Customer Visit,15.0,Средняя,Длинная
6,2016-02-01 12:10:00,2016-02-01 12:43:00,Business,Chapel Hill,Cary,45008.0,Customer Visit,33.0,Длинная,Средняя
7,2016-02-04 09:37:00,2016-02-04 10:09:00,Business,Morrisville,Cary,45116.0,Meal/Entertain,32.0,Длинная,Длинная
8,2016-02-07 18:03:00,2016-02-07 18:17:00,Business,Apex,Cary,45112.0,Customer Visit,14.0,Средняя,Длинная
9,2016-02-07 20:22:00,2016-02-07 20:40:00,Business,Morrisville,Cary,44932.0,Meeting,18.0,Средняя,Короткая


## Выводы

В ходе лабораторной работы с датасетом `drivers.csv` (161 запись, журнал поездок на такси) освоены методы агрегации и группировки данных библиотеки `pandas`.

- Выполнена **предварительная обработка**: переименование столбцов (`CATEGORY*` - `CATEGORY`, `PURPOSEroute` - `PURPOSE_ROUTE`), преобразование дат в `datetime`, расчёт длительности поездок (`DURATION`), исправление неявных дубликатов (`BUSINESS` - `Business`,
`MEETING` - `Meeting`), заполнение 77 пропусков в `PURPOSE_ROUTE` значением `'не определена'`.
- **Задание 1:** `groupby(['CATEGORY', 'PURPOSE_ROUTE'])` - 150 деловых и 11 личных поездок; наиболее частые цели деловых поездок - Meal/Entertain (36) и Customer Visit (30).
- **Задание 2:** `groupby(['CATEGORY', 'START'])` - самая популярная точка отправления для Business-поездок - Morrisville (70 поездок); результат сохранён в DataFrame с переименованным столбцом `count`, отсортирован по возрастанию.
- **Задание 3:** `pivot_table` - среднее `MILES` по `PURPOSE_ROUTE`; наибольшее значение у Temporary Site (45 063), наименьшее - у Meeting (34 647).
- **Задание 4:** `pivot_table` - среднее `MILES`, строки = `START`, столбцы = `CATEGORY`; отсортировано по убыванию индекса `START`. Высокие значения одометра соответствуют поздним поездкам года.
- **Задание 5:** создан столбец `DURATION_CATEGORY` (границы по Q1=9 мин и Q3=22 мин);
  `pivot_table` - среднее `MILES` по категории длительности и цели маршрута.
- **Задание 6:** создан столбец `MILES_CATEGORY` (границы по 33-му и 66-му перцентилям: 44 959 и 45 050); `groupby` - минимальная и максимальная длительность поездки по категории пройденных миль. Наибольший разброс длительности - у категории «Короткая» (1–206 мин).